In [2]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import collections
import copy

%matplotlib inline
path = "/home/jrosa/AGH_FILES/ZAW-2025S/lab02_cfd/pedestrian/"

In [31]:
prev_IG = cv2.imread(path + "input/in000001.jpg")
prev_IG = cv2.cvtColor(prev_IG , cv2.COLOR_BGR2GRAY)
theshold = 18
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

T_P = 0
T_N = 0
F_P = 0
F_N = 0

class BackgroungModel:
    def __init__(self):
        self.background_buff = collections.deque(maxlen = 60)
        self.learning_background = None
        self.BackgroundMOG2 = cv2.createBackgroundSubtractorMOG2(history=10)
        self.BackgroundKNN = cv2.createBackgroundSubtractorKNN(history=10)

    def circular_mean(self, I):
        self.background_buff.append(I)
        return np.mean(self.background_buff, axis=0)
    
    def circular_median(self, I):
        self.background_buff.append(I)
        return np.median(self.background_buff, axis=0)
    
    def learning_mean(self, I, alpha = 0.2):
        if self.learning_background is None:
            self.learning_background = I
        self.learning_background = alpha*I.astype(np.float64) + (1 - alpha)*self.learning_background.astype(np.float64)
        return self.learning_background
    
    def learning_median(self, I, alpha = 0.2):
        if self.learning_background is None:
            self.learning_background = I
        self.learning_background = self.learning_background + alpha*(0 > np.subtract(self.learning_background, I))
        self.learning_background = self.learning_background - alpha*(0 < np.subtract(self.learning_background, I))
        return self.learning_background
    
    def conservative_mean(self, I, IB, alpha = 0.1):
        if self.learning_background is None:
            self.learning_background = copy.copy(I)
            print("Initialized")
            return I
        mask = lambda Ip, Im : cv2.bitwise_and(Ip.astype(np.uint8), Ip.astype(np.uint8), mask=255*Im.astype(np.uint8))
        new_background = alpha * mask(I, IB).astype(np.float64) + (1 - alpha) * mask(self.learning_background, IB).astype(np.float64)
        self.learning_background = new_background + mask(self.learning_background, np.logical_not(IB))
        
        return self.learning_background.astype(np.uint8)
    
    def conservative_median(self, I, IB, alpha = 0.05):
        if self.learning_background is None:
            self.learning_background = copy.copy(I)
            print("Initialized")
            return I
        mask = lambda Ip, Im : cv2.bitwise_and(Ip.astype(np.uint8), Ip.astype(np.uint8), mask=255*Im.astype(np.uint8))
        new_background = mask(self.learning_background + alpha*(0 > np.subtract(self.learning_background, I)), IB)
        new_background = new_background - mask(alpha*(0 < np.subtract(self.learning_background, I)), IB)
        self.learning_background = new_background + mask(self.learning_background, np.logical_not(IB))
        
        return self.learning_background.astype(np.uint8)
    
    def opencv_MOG2(self, I):
        return self.BackgroundMOG2.apply(I)
    
    def opencv_KNN(self, I):
        return self.BackgroundKNN.apply(I)

        


    
backgroundModel = BackgroungModel()
I_close = np.zeros(prev_IG.shape)



for i in range(300, 1100):
    I = cv2.imread(path + "input/in%06d.jpg" % i)
    I_GROUND = cv2.imread(path + "groundtruth/gt%06d.png" % i)
    I_GROUND = cv2.cvtColor(I_GROUND, cv2.COLOR_BGR2GRAY)

    I_background = backgroundModel.conservative_mean(I, np.logical_not(I_close)).astype(np.uint8)

    IG = cv2.cvtColor(I, cv2.COLOR_BGR2GRAY)

    # I_background = backgroundModel.learning_median(I).astype(np.uint8)
    
    IBG = cv2.cvtColor(I_background.astype(np.uint8), cv2.COLOR_BGR2GRAY)

    I_diff = cv2.absdiff(IG, IBG)

    I_diff = (255*( I_diff > theshold)).astype(np.uint8)

    I_diff = backgroundModel.opencv_MOG2(I)

    I_diff = cv2.medianBlur(I_diff, 5)

    I_open = cv2.morphologyEx(I_diff, cv2.MORPH_OPEN, kernel)
    I_close = cv2.morphologyEx(I_open, cv2.MORPH_CLOSE, kernel_big)
    I_close = cv2.morphologyEx(I_close, cv2.MORPH_CLOSE, kernel_big)
    I_close = cv2.morphologyEx(I_close, cv2.MORPH_CLOSE, kernel_big)

    retval, labels, stats, centroids = cv2.connectedComponentsWithStats(I_close)

    # cv2.imshow("Stream", cv2.cvtColor(I_close, cv2.COLOR_GRAY2BGR))
    cv2.waitKey(5)

    # Display the labels image with appropriate scaling
    # cv2.imshow("Labels", np.uint8(labels / retval * 255))

    I_VIS = np.copy(I)

    # Check if there are any detected objects
    if stats.shape[0] > 1:
        tab = stats[1:, 4]  # Extract the area column excluding the first element
        pi = np.argmax(tab)  # Find the index of the largest object
        pi = pi + 1  # Adjust index to match 'stats'
        
        # Draw bounding box around the largest object
        cv2.rectangle(I_VIS, (stats[pi, 0], stats[pi, 1]), 
                    (stats[pi, 0] + stats[pi, 2], stats[pi, 1] + stats[pi, 3]), 
                    (255, 0, 0), 2)
        
        # Display area of the largest object
        cv2.putText(I_VIS, "%.2f" % stats[pi, 4], 
                    (stats[pi, 0], stats[pi, 1]), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0))
        
        # Display index of the largest object at its centroid
        cv2.putText(I_VIS, "%d" % pi, 
                    (int(centroids[pi, 0]), int(centroids[pi, 1])), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0))
        
        true_positive_mask = np.logical_and((I_close == 255), (I_GROUND == 255))
        true_positive_sum = np.sum(true_positive_mask)
        T_P = T_P + true_positive_sum

        true_negative_mask = np.logical_and((I_close == 0), (I_GROUND == 0))
        true_negative_sum = np.sum(true_negative_mask)
        T_N = T_N + true_negative_sum

        false_positive_mask = np.logical_and((I_close == 255), (I_GROUND == 0))
        false_positive_sum = np.sum(false_positive_mask)
        F_P = F_P + false_positive_sum

        false_negative_mask = np.logical_and((I_close == 0), (I_GROUND == 255))
        false_negative_sum = np.sum(false_negative_mask)
        F_N = F_N + false_negative_sum

    cv2.imshow("Labels", I_VIS)
    cv2.imshow("BackGround", I_background)

    prev_IG = IG

cv2.destroyAllWindows()  # close all windows


precision = T_P / (T_P + F_P) if (T_P + F_P) > 0 else 0
recall = T_P / (T_P + F_N) if (T_P + F_N) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0


print("Presision: ", precision)
print("recall: ", recall)
print("F1: ", f1_score)

Initialized
Presision:  0
recall:  0.0
F1:  0
